In [1]:
%pip install crunch-cli --upgrade --quiet --progress-bar off
!crunch setup-notebook structural-break-real-time BgD6OGyfkCKpr9waHz56h0tj

crunch-cli, version 12.0.2
main.py: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/submissions/78334/main.py (51177 bytes)
notebook.ipynb: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/submissions/78334/notebook.ipynb (65946 bytes)
requirements.txt: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/submissions/78334/requirements.txt (196 bytes)
data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_train.parquet (8356193 bytes)
data/y_test.reduced.parquet: download from https:crunchdao--compet

In [2]:
import math
import os
from typing import Iterable, List, Optional, Tuple

# Import your dependencies.
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


import crunch

# Load the Crunch Toolings (data loader, local tester, submitter).
crunch_tools = crunch.load_notebook()


# Load the data.
train_data, test_data = crunch_tools.load_data()


from collections import deque
import math
import numpy as np

def _win_stats_blocks(zh, w):
    n = len(zh)
    if n < 2*w:
        return None
    step = max(w // 2, 1)
    lvars, kss, skews, ac1s, means = [], [], [], [], []
    qs = np.quantile(zh, np.linspace(0.1, 0.9, 9))
    for start in range(0, n - w + 1, step):
        seg = zh[start:start+w]
        m = seg.mean(); v = max(seg.var(), 1e-12)
        lvars.append(math.log(v))
        zn = (seg - m)/math.sqrt(v)
        skews.append(float((zn**3).mean()))
        a = seg - m
        ac1s.append(float(np.dot(a[1:], a[:-1]) / max(np.dot(a, a), 1e-12)))
        emp = np.searchsorted(np.sort(seg), qs, side="right") / w
        kss.append(float(np.max(np.abs(emp - np.linspace(0.1, 0.9, 9)))))
        means.append(m)
    def musd(v):
        v = np.asarray(v); return float(v.mean()), max(float(v.std()), 1e-6)
    return {"lvar": musd(lvars), "ks": musd(kss), "skew": musd(skews),
            "ac1": musd(ac1s), "mean": musd(means)}

def _fit_ar_p(z, p):
    n = len(z)
    if p == 0:
        return np.zeros(0), max(float(z.var()), 1e-6), n
    if n < 20 * p:
        return None
    y = z[p:]
    cols = [z[p-1-i : n-1-i] for i in range(p)]
    X = np.column_stack(cols)
    XtX = X.T @ X + 1e-6 * np.eye(p)
    phi = np.linalg.solve(XtX, X.T @ y)
    resid = y - X @ phi
    return phi, max(float(resid.var()), 1e-6), len(y)

def _select_ar_order(z, max_p=4):
    best_bic = None
    best = (0, np.zeros(0), max(float(z.var()), 1e-6))
    for p in range(0, max_p + 1):
        r = _fit_ar_p(z, p)
        if r is None:
            continue
        phi, rv, m = r
        bic = m * math.log(rv) + (p + 1) * math.log(max(m, 2))
        if best_bic is None or bic < best_bic:
            best_bic = bic
            best = (p, phi, rv)
    return best

class _TrailingAR:
    """Rolling-window AR(p) refit, compared against the history fit."""
    __slots__ = ("p", "W", "buf", "hist_phi", "hist_rv")
    def __init__(self, p, W, hist_phi, hist_rv):
        self.p = max(p, 1)
        self.W = W
        self.buf = deque(maxlen=W)
        hp = np.asarray(hist_phi, dtype=np.float64)
        if len(hp) < self.p:
            hp = np.zeros(self.p)
        self.hist_phi = hp
        self.hist_rv = max(hist_rv, 1e-9)
    def update(self, z):
        self.buf.append(z)
        n = len(self.buf)
        if n < max(self.W, 10 * self.p):
            return 0.0, 0.0
        seg = np.asarray(self.buf)
        p = self.p
        y = seg[p:]
        cols = [seg[p-1-i : n-1-i] for i in range(p)]
        X = np.column_stack(cols)
        try:
            phi = np.linalg.solve(X.T @ X + 1e-6 * np.eye(p), X.T @ y)
        except Exception:
            return 0.0, 0.0
        r = y - X @ phi
        rv = max(float(r.var()), 1e-9)
        return (float(np.linalg.norm(phi - self.hist_phi[:p])),
                math.log(rv / self.hist_rv))

class StreamingFeatures:
    def __init__(self, x_hist, ar_order=None):
        x = np.asarray(x_hist, dtype=np.float64)
        self.mu_h = float(x.mean()); self.sd_h = max(float(x.std(ddof=1)), 1e-8)
        xc = x - self.mu_h
        den_h = max(np.dot(xc, xc), 1e-12)
        self.ac1_h = float(np.dot(xc[1:], xc[:-1]) / den_h)
        self.ac2_h = float(np.dot(xc[2:], xc[:-2]) / den_h)
        self.ac5_h = float(np.dot(xc[5:], xc[:-5]) / den_h)
        zh = xc / self.sd_h
        self.qs = np.quantile(zh, np.linspace(0.1, 0.9, 9))
        self.skew_h = float((zh**3).mean())
        self.kurt_h = float((zh**4).mean()) - 3.0
        self.cal100 = _win_stats_blocks(zh, 100)
        self.cal20 = _win_stats_blocks(zh, 20)
        self.t = 0; self.sum = 0.0; self.sumsq = 0.0
        self.cusum_pos = 0.0; self.cusum_neg = 0.0; self.n_tail = 0
        self.prev_z = None; self.ac1_num = 0.0; self.ac1_den = 0.0
        self.w20 = deque(); self.s20 = 0.0; self.q20 = 0.0
        self.w100 = deque(maxlen=100); self.s100 = 0.0; self.q100 = 0.0
        self.pz = [0.0]; self.pz2 = [0.0]; self.pz3 = [0.0]; self.pt = [0]

        if ar_order is None:
            p, phi, rv = _select_ar_order(zh, 4)
        else:
            p = ar_order
            r = _fit_ar_p(zh, p)
            if r is None:
                p, phi, rv = _select_ar_order(zh, 4)
            else:
                phi, rv, _ = r
        self.p = p
        self.phi = np.asarray(phi, dtype=np.float64)
        self.rv = rv
        self.srv = math.sqrt(rv)
        self.lags = [0.0] * max(p, 1)
        for i in range(p):
            self.lags[i] = zh[-1 - i] if len(zh) > i else 0.0

        self.ll_cum = 0.0
        self.r_sum = 0.0; self.r_sq = 0.0; self.r_n = 0
        self.r_prev = None; self.r_ac_num = 0.0; self.r_ac_den = 0.0
        self.rw = deque(maxlen=50); self.rw_s = 0.0; self.rw_q = 0.0
        self.r_cusum = 0.0
        self.ew_var = 1.0; self.g_cusum = 0.0; self.g_ll = 0.0; self.g_n = 0
        self.u3 = 0.0; self.u4 = 3.0

    def update(self, x):
        self.t += 1
        z = (x - self.mu_h) / self.sd_h
        k = 0.5
        self.cusum_pos = max(0.0, self.cusum_pos + z - k)
        self.cusum_neg = max(0.0, self.cusum_neg - z - k)
        cusum = max(self.cusum_pos, self.cusum_neg)
        self.sum += x; self.sumsq += x * x
        if self.t > 1:
            m = self.sum / self.t
            var_o = max((self.sumsq - self.t * m * m) / (self.t - 1), 1e-12) / self.sd_h ** 2
        else:
            var_o = 1.0
        var_ratio = math.log(var_o)
        if abs(z) > 2.0: self.n_tail += 1
        tail_frac = self.n_tail / self.t
        if self.prev_z is not None:
            self.ac1_num += z * self.prev_z; self.ac1_den += z * z
        ac1_o = self.ac1_num / self.ac1_den if self.ac1_den > 1e-12 else 0.0
        self.prev_z = z

        self.w20.append(z); self.s20 += z; self.q20 += z * z
        if len(self.w20) > 20:
            old = self.w20.popleft(); self.s20 -= old; self.q20 -= old * old
        n20 = len(self.w20); m20 = self.s20 / n20
        v20 = max(self.q20 / n20 - m20 * m20, 1e-12)

        if len(self.w100) == 100:
            old = self.w100[0]; self.s100 -= old; self.q100 -= old * old
        self.w100.append(z); self.s100 += z; self.q100 += z * z
        n100 = len(self.w100); m100 = self.s100 / n100
        v100 = max(self.q100 / n100 - m100 * m100, 1e-12)
        arr = np.asarray(self.w100)
        zn = (arr - m100) / math.sqrt(v100)
        skew_raw = float((zn**3).mean())
        w100_skew = skew_raw - self.skew_h
        w100_kurt = float((zn**4).mean()) - 3.0 - self.kurt_h
        a = arr - m100
        den = max(np.dot(a, a), 1e-12)
        ac1_100 = float(np.dot(a[1:], a[:-1]) / den) if n100 >= 5 else self.ac1_h
        w100_ac1 = ac1_100 - self.ac1_h
        w100_ac2 = (float(np.dot(a[2:], a[:-2]) / den) if n100 >= 7 else self.ac2_h) - self.ac2_h
        w100_ac5 = (float(np.dot(a[5:], a[:-5]) / den) if n100 >= 12 else self.ac5_h) - self.ac5_h
        emp = np.searchsorted(np.sort(arr), self.qs, side="right") / n100
        ks100 = float(np.max(np.abs(emp - np.linspace(0.1, 0.9, 9))))
        signs = np.sign(arr)
        flips = float(np.mean(signs[1:] * signs[:-1] < 0)) if n100 >= 3 else 0.5

        c = self.cal100
        if c is not None and n100 >= 50:
            cal_lvar = (math.log(v100) - c["lvar"][0]) / c["lvar"][1]
            cal_ks   = (ks100 - c["ks"][0]) / c["ks"][1]
            cal_skew = (skew_raw - c["skew"][0]) / c["skew"][1]
            cal_ac1  = (ac1_100 - c["ac1"][0]) / c["ac1"][1]
        else:
            cal_lvar = cal_ks = cal_skew = cal_ac1 = 0.0
        c2 = self.cal20
        if c2 is not None and n20 >= 10:
            cal_mean20 = (m20 - c2["mean"][0]) / c2["mean"][1]
        else:
            cal_mean20 = 0.0

        self.pz.append(self.pz[-1] + z)
        self.pz2.append(self.pz2[-1] + z * z)
        self.pz3.append(self.pz3[-1] + z ** 3)
        self.pt.append(self.pt[-1] + (1 if abs(z) > 2 else 0))
        t = self.t
        scan_mean = scan_lvar = scan_skew = scan_tail = 0.0
        if t >= 8:
            for f in (0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85):
                s = max(2, int(t * f))
                if s >= t - 1: continue
                n1, n2 = s, t - s
                m1 = self.pz[s] / n1; m2 = (self.pz[t] - self.pz[s]) / n2
                v1 = max(self.pz2[s] / n1 - m1 * m1, 1e-12)
                v2 = max((self.pz2[t] - self.pz2[s]) / n2 - m2 * m2, 1e-12)
                dmean = abs(m2 - m1) / math.sqrt(v1 / n1 + v2 / n2)
                dlvar = abs(math.log(v2 / v1))
                mu3_1 = self.pz3[s] / n1 - 3 * m1 * v1 - m1 ** 3
                mu3_2 = (self.pz3[t] - self.pz3[s]) / n2 - 3 * m2 * v2 - m2 ** 3
                dskew = abs(mu3_2 / v2 ** 1.5 - mu3_1 / v1 ** 1.5)
                dtail = abs((self.pt[t] - self.pt[s]) / n2 - self.pt[s] / n1)
                if dmean > scan_mean: scan_mean = dmean
                if dlvar > scan_lvar: scan_lvar = dlvar
                if dskew > scan_skew: scan_skew = dskew
                if dtail > scan_tail: scan_tail = dtail

        pred = 0.0
        for j in range(self.p):
            pred += self.phi[j] * self.lags[j]
        u = (z - pred) / self.srv
        if self.p > 0:
            for j in range(self.p - 1, 0, -1):
                self.lags[j] = self.lags[j-1]
            self.lags[0] = z
        self.ll_cum += 0.5 * (u * u - 1.0)
        self.r_n += 1; self.r_sum += u; self.r_sq += u * u
        r_var = self.r_sq / self.r_n - (self.r_sum / self.r_n) ** 2 if self.r_n > 1 else 1.0
        if self.r_prev is not None:
            self.r_ac_num += u * self.r_prev; self.r_ac_den += u * u
        r_ac = self.r_ac_num / self.r_ac_den if self.r_ac_den > 1e-12 else 0.0
        self.r_prev = u
        if len(self.rw) == 50:
            old = self.rw[0]; self.rw_s -= old; self.rw_q -= old * old
        self.rw.append(u); self.rw_s += u; self.rw_q += u * u
        nw = len(self.rw); mw = self.rw_s / nw
        vw = max(self.rw_q / nw - mw * mw, 1e-12)
        self.r_cusum = max(0.0, self.r_cusum + u - 0.5)
        ll_norm = self.ll_cum / math.sqrt(max(self.r_n, 1))

        g = u / math.sqrt(max(self.ew_var, 1e-6))
        self.ew_var = 0.94 * self.ew_var + 0.06 * u * u
        self.g_n += 1
        self.g_ll += 0.5 * (g * g - 1.0)
        self.g_cusum = max(0.0, self.g_cusum + abs(g) - 0.8)
        self.u3 = 0.97 * self.u3 + 0.03 * (u ** 3)
        self.u4 = 0.97 * self.u4 + 0.03 * (u ** 4)
        g_ll_norm = self.g_ll / math.sqrt(max(self.g_n, 1))
        g_cusum_n = self.g_cusum / math.sqrt(max(self.g_n, 1))

        return (self.t, cusum, var_ratio, tail_frac, ac1_o - self.ac1_h, abs(z),
                m20 * math.sqrt(n20), math.log(v20),
                m100 * math.sqrt(n100), math.log(v100),
                w100_skew, w100_kurt, w100_ac1, ks100, flips,
                cal_lvar, cal_ks, cal_skew, cal_ac1, cal_mean20,
                w100_ac2, w100_ac5, scan_mean, scan_lvar, scan_skew, scan_tail,
                ll_norm, math.log(max(r_var, 1e-12)), r_ac,
                math.log(vw), mw * math.sqrt(nw), self.r_cusum,
                g_ll_norm, g_cusum_n, math.log(max(self.ew_var, 1e-6)),
                self.u3, self.u4 - 3.0)


%pip install catboost --quiet

!pip install numba --quiet


import math
import numpy as np
from numba import njit

@njit(cache=False, fastmath=False)
def _run_series(o, mu_h, sd_h, ac1_h, ac2_h, ac5_h, skew_h, kurt_h,
                qs, grid, cal100, has_cal100, cal20mean, has_cal20,
                p, phi, srv, lags0):
    n_o = o.shape[0]
    out = np.empty((n_o, 37), dtype=np.float64)
    tsum = 0.0; tsumsq = 0.0
    cusum_pos = 0.0; cusum_neg = 0.0; n_tail = 0
    have_prev = False; prev_z = 0.0
    ac1_num = 0.0; ac1_den = 0.0
    w20 = np.zeros(20); n20 = 0; h20 = 0; s20 = 0.0; q20 = 0.0
    w100 = np.zeros(100); n100 = 0; h100 = 0; s100 = 0.0; q100 = 0.0
    buf = np.empty(100); srt = np.empty(100)
    pz = np.zeros(n_o + 1); pz2 = np.zeros(n_o + 1)
    pz3 = np.zeros(n_o + 1); pt = np.zeros(n_o + 1, dtype=np.int64)
    fracs = np.array([0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85])
    lags = lags0.copy()
    ll_cum = 0.0; r_sum = 0.0; r_sq = 0.0; r_n = 0
    have_rprev = False; r_prev = 0.0; r_ac_num = 0.0; r_ac_den = 0.0
    rw = np.zeros(50); nrw = 0; hrw = 0; rw_s = 0.0; rw_q = 0.0
    r_cusum = 0.0; ew_var = 1.0; g_cusum = 0.0; g_ll = 0.0; g_n = 0
    u3 = 0.0; u4 = 3.0

    for ti in range(n_o):
        x = o[ti]; t = ti + 1
        z = (x - mu_h) / sd_h
        a1 = cusum_pos + z - 0.5
        cusum_pos = a1 if a1 > 0.0 else 0.0
        a2 = cusum_neg - z - 0.5
        cusum_neg = a2 if a2 > 0.0 else 0.0
        cusum = cusum_pos if cusum_pos > cusum_neg else cusum_neg
        tsum += x; tsumsq += x * x
        if t > 1:
            m = tsum / t
            vo = (tsumsq - t * m * m) / (t - 1)
            if vo < 1e-12: vo = 1e-12
            var_o = vo / (sd_h * sd_h)
        else:
            var_o = 1.0
        var_ratio = math.log(var_o)
        az = z if z >= 0.0 else -z
        if az > 2.0: n_tail += 1
        tail_frac = n_tail / t
        if have_prev:
            ac1_num += z * prev_z; ac1_den += z * z
        ac1_o = ac1_num / ac1_den if ac1_den > 1e-12 else 0.0
        prev_z = z; have_prev = True

        if n20 == 20:
            old = w20[h20]; s20 -= old; q20 -= old * old
            w20[h20] = z; h20 = (h20 + 1) % 20
        else:
            w20[(h20 + n20) % 20] = z; n20 += 1
        s20 += z; q20 += z * z
        m20 = s20 / n20
        v20 = q20 / n20 - m20 * m20
        if v20 < 1e-12: v20 = 1e-12

        if n100 == 100:
            old = w100[h100]; s100 -= old; q100 -= old * old
            w100[h100] = z; h100 = (h100 + 1) % 100
        else:
            w100[(h100 + n100) % 100] = z; n100 += 1
        s100 += z; q100 += z * z
        for i in range(n100):
            buf[i] = w100[(h100 + i) % 100]
        m100 = s100 / n100
        v100 = q100 / n100 - m100 * m100
        if v100 < 1e-12: v100 = 1e-12

        sv = math.sqrt(v100); s3 = 0.0; s4 = 0.0
        for i in range(n100):
            zz = (buf[i] - m100) / sv
            z2 = zz * zz
            s3 += z2 * zz; s4 += z2 * z2
        skew_raw = s3 / n100
        w100_skew = skew_raw - skew_h
        w100_kurt = s4 / n100 - 3.0 - kurt_h

        den = 0.0
        for i in range(n100):
            d0 = buf[i] - m100
            den += d0 * d0
        if den < 1e-12: den = 1e-12

        if n100 >= 5:
            acc = 0.0
            for i in range(1, n100):
                acc += (buf[i] - m100) * (buf[i - 1] - m100)
            ac1_100 = acc / den
        else:
            ac1_100 = ac1_h
        w100_ac1 = ac1_100 - ac1_h

        if n100 >= 7:
            acc = 0.0
            for i in range(2, n100):
                acc += (buf[i] - m100) * (buf[i - 2] - m100)
            w100_ac2 = acc / den - ac2_h
        else:
            w100_ac2 = 0.0

        if n100 >= 12:
            acc = 0.0
            for i in range(5, n100):
                acc += (buf[i] - m100) * (buf[i - 5] - m100)
            w100_ac5 = acc / den - ac5_h
        else:
            w100_ac5 = 0.0

        for i in range(n100):
            srt[i] = buf[i]
        sub = srt[:n100]
        sub.sort()
        ks100 = 0.0
        for j in range(9):
            qv = qs[j]; lo = 0; hi = n100
            while lo < hi:
                mid = (lo + hi) // 2
                if sub[mid] <= qv: lo = mid + 1
                else: hi = mid
            dd = lo / n100 - grid[j]
            if dd < 0.0: dd = -dd
            if dd > ks100: ks100 = dd

        if n100 >= 3:
            fl = 0
            for i in range(1, n100):
                aa = buf[i - 1]; bb = buf[i]
                sa = 1.0 if aa > 0.0 else (-1.0 if aa < 0.0 else 0.0)
                sb = 1.0 if bb > 0.0 else (-1.0 if bb < 0.0 else 0.0)
                if sa * sb < 0.0: fl += 1
            flips = fl / (n100 - 1)
        else:
            flips = 0.5

        if has_cal100 and n100 >= 50:
            cal_lvar = (math.log(v100) - cal100[0]) / cal100[1]
            cal_ks = (ks100 - cal100[2]) / cal100[3]
            cal_skew = (skew_raw - cal100[4]) / cal100[5]
            cal_ac1 = (ac1_100 - cal100[6]) / cal100[7]
        else:
            cal_lvar = 0.0; cal_ks = 0.0; cal_skew = 0.0; cal_ac1 = 0.0

        if has_cal20 and n20 >= 10:
            cal_mean20 = (m20 - cal20mean[0]) / cal20mean[1]
        else:
            cal_mean20 = 0.0

        pz[t] = pz[t - 1] + z
        pz2[t] = pz2[t - 1] + z * z
        pz3[t] = pz3[t - 1] + z * z * z
        pt[t] = pt[t - 1] + (1 if az > 2.0 else 0)

        scan_mean = 0.0; scan_lvar = 0.0; scan_skew = 0.0; scan_tail = 0.0
        if t >= 8:
            for fi in range(8):
                s = int(t * fracs[fi])
                if s < 2: s = 2
                if s >= t - 1: continue
                n1 = s; n2 = t - s
                m1 = pz[s] / n1; m2 = (pz[t] - pz[s]) / n2
                v1 = pz2[s] / n1 - m1 * m1
                if v1 < 1e-12: v1 = 1e-12
                v2 = (pz2[t] - pz2[s]) / n2 - m2 * m2
                if v2 < 1e-12: v2 = 1e-12
                dmean = (m2 - m1) / math.sqrt(v1 / n1 + v2 / n2)
                if dmean < 0.0: dmean = -dmean
                dlvar = math.log(v2 / v1)
                if dlvar < 0.0: dlvar = -dlvar
                mu3_1 = pz3[s] / n1 - 3.0 * m1 * v1 - m1 * m1 * m1
                mu3_2 = (pz3[t] - pz3[s]) / n2 - 3.0 * m2 * v2 - m2 * m2 * m2
                dskew = mu3_2 / (v2 ** 1.5) - mu3_1 / (v1 ** 1.5)
                if dskew < 0.0: dskew = -dskew
                dtail = (pt[t] - pt[s]) / n2 - pt[s] / n1
                if dtail < 0.0: dtail = -dtail
                if dmean > scan_mean: scan_mean = dmean
                if dlvar > scan_lvar: scan_lvar = dlvar
                if dskew > scan_skew: scan_skew = dskew
                if dtail > scan_tail: scan_tail = dtail

        pred = 0.0
        for j in range(p):
            pred += phi[j] * lags[j]
        u = (z - pred) / srv
        if p > 0:
            for j in range(p - 1, 0, -1):
                lags[j] = lags[j - 1]
            lags[0] = z

        ll_cum += 0.5 * (u * u - 1.0)
        r_n += 1; r_sum += u; r_sq += u * u
        r_var = r_sq / r_n - (r_sum / r_n) * (r_sum / r_n) if r_n > 1 else 1.0
        if have_rprev:
            r_ac_num += u * r_prev; r_ac_den += u * u
        r_ac = r_ac_num / r_ac_den if r_ac_den > 1e-12 else 0.0
        r_prev = u; have_rprev = True

        if nrw == 50:
            old = rw[hrw]; rw_s -= old; rw_q -= old * old
            rw[hrw] = u; hrw = (hrw + 1) % 50
        else:
            rw[(hrw + nrw) % 50] = u; nrw += 1
        rw_s += u; rw_q += u * u
        mw = rw_s / nrw
        vw = rw_q / nrw - mw * mw
        if vw < 1e-12: vw = 1e-12

        rc = r_cusum + u - 0.5
        r_cusum = rc if rc > 0.0 else 0.0
        ll_norm = ll_cum / math.sqrt(r_n)

        evd = ew_var if ew_var > 1e-6 else 1e-6
        g = u / math.sqrt(evd)
        ew_var = 0.94 * ew_var + 0.06 * u * u
        g_n += 1
        g_ll += 0.5 * (g * g - 1.0)
        ag = g if g >= 0.0 else -g
        gc = g_cusum + ag - 0.8
        g_cusum = gc if gc > 0.0 else 0.0
        u3 = 0.97 * u3 + 0.03 * (u * u * u)
        u4 = 0.97 * u4 + 0.03 * (u * u * u * u)
        g_ll_norm = g_ll / math.sqrt(g_n)
        g_cusum_n = g_cusum / math.sqrt(g_n)

        rv_c = r_var if r_var > 1e-12 else 1e-12
        ev_c = ew_var if ew_var > 1e-6 else 1e-6

        out[ti, 0] = t;  out[ti, 1] = cusum;  out[ti, 2] = var_ratio
        out[ti, 3] = tail_frac;  out[ti, 4] = ac1_o - ac1_h;  out[ti, 5] = az
        out[ti, 6] = m20 * math.sqrt(n20);  out[ti, 7] = math.log(v20)
        out[ti, 8] = m100 * math.sqrt(n100);  out[ti, 9] = math.log(v100)
        out[ti, 10] = w100_skew;  out[ti, 11] = w100_kurt;  out[ti, 12] = w100_ac1
        out[ti, 13] = ks100;  out[ti, 14] = flips
        out[ti, 15] = cal_lvar;  out[ti, 16] = cal_ks;  out[ti, 17] = cal_skew
        out[ti, 18] = cal_ac1;  out[ti, 19] = cal_mean20
        out[ti, 20] = w100_ac2;  out[ti, 21] = w100_ac5
        out[ti, 22] = scan_mean;  out[ti, 23] = scan_lvar
        out[ti, 24] = scan_skew;  out[ti, 25] = scan_tail
        out[ti, 26] = ll_norm;  out[ti, 27] = math.log(rv_c);  out[ti, 28] = r_ac
        out[ti, 29] = math.log(vw);  out[ti, 30] = mw * math.sqrt(nrw)
        out[ti, 31] = r_cusum;  out[ti, 32] = g_ll_norm;  out[ti, 33] = g_cusum_n
        out[ti, 34] = math.log(ev_c);  out[ti, 35] = u3;  out[ti, 36] = u4 - 3.0
    return out


class StreamingFeaturesJIT:
    def __init__(self, x_hist, ar_order=None):
        x = np.asarray(x_hist, dtype=np.float64)
        self.mu_h = float(x.mean())
        self.sd_h = max(float(x.std(ddof=1)), 1e-8)
        xc = x - self.mu_h
        den_h = max(np.dot(xc, xc), 1e-12)
        self.ac1_h = float(np.dot(xc[1:], xc[:-1]) / den_h)
        self.ac2_h = float(np.dot(xc[2:], xc[:-2]) / den_h)
        self.ac5_h = float(np.dot(xc[5:], xc[:-5]) / den_h)
        zh = xc / self.sd_h
        self.qs = np.quantile(zh, np.linspace(0.1, 0.9, 9))
        self.grid = np.linspace(0.1, 0.9, 9)
        self.skew_h = float((zh ** 3).mean())
        self.kurt_h = float((zh ** 4).mean()) - 3.0
        c = _win_stats_blocks(zh, 100)
        if c is None:
            self.cal100 = np.zeros(8); self.has_cal100 = False
        else:
            self.cal100 = np.array([c["lvar"][0], c["lvar"][1], c["ks"][0], c["ks"][1],
                                    c["skew"][0], c["skew"][1], c["ac1"][0], c["ac1"][1]])
            self.has_cal100 = True
        c2 = _win_stats_blocks(zh, 20)
        if c2 is None:
            self.cal20 = np.zeros(2); self.has_cal20 = False
        else:
            self.cal20 = np.array([c2["mean"][0], c2["mean"][1]]); self.has_cal20 = True
        if ar_order is None:
            p, phi, rv = _select_ar_order(zh, 4)
        else:
            p = ar_order
            r = _fit_ar_p(zh, p)
            if r is None:
                p, phi, rv = _select_ar_order(zh, 4)
            else:
                phi, rv, _ = r
        self.p = int(p)
        self.phi = np.asarray(phi, dtype=np.float64)
        if self.phi.shape[0] == 0:
            self.phi = np.zeros(1)
        self.rv = rv
        self.srv = math.sqrt(rv)
        lags0 = np.zeros(max(self.p, 1))
        for i in range(self.p):
            lags0[i] = zh[-1 - i] if len(zh) > i else 0.0
        self.lags0 = lags0

    def run(self, x_online):
        o = np.asarray(x_online, dtype=np.float64)
        return _run_series(o, self.mu_h, self.sd_h, self.ac1_h, self.ac2_h,
                           self.ac5_h, self.skew_h, self.kurt_h,
                           self.qs, self.grid, self.cal100, self.has_cal100,
                           self.cal20, self.has_cal20,
                           self.p, self.phi, self.srv, self.lags0)


# ═══════════════════════════════════════════════════════════
# CELL 3 of 5 — NULL CALIBRATION  ·  KEEP THIS CELL
# Order: 1 class → 2 jit → 3 THIS → 4 train → 5 infer
# Widths are defined INSIDE each function: bare top-level
# assignments get stripped by the notebook→main.py converter.
# ═══════════════════════════════════════════════════════════
import numpy as np, math
from numba import njit

@njit(cache=False)
def _scan_stats_at(z, start, n):
    pz = np.zeros(n + 1); pz2 = np.zeros(n + 1)
    pz3 = np.zeros(n + 1); pt = np.zeros(n + 1, dtype=np.int64)
    for i in range(n):
        v = z[start + i]
        pz[i+1] = pz[i] + v
        pz2[i+1] = pz2[i] + v * v
        pz3[i+1] = pz3[i] + v * v * v
        pt[i+1] = pt[i] + (1 if (v if v >= 0 else -v) > 2.0 else 0)
    fracs = np.array([0.15,0.25,0.35,0.45,0.55,0.65,0.75,0.85])
    t = n; sm = 0.0; sl = 0.0; sk = 0.0; st = 0.0
    for fi in range(8):
        s = int(t * fracs[fi])
        if s < 2: s = 2
        if s >= t - 1: continue
        n1 = s; n2 = t - s
        m1 = pz[s]/n1; m2 = (pz[t]-pz[s])/n2
        v1 = pz2[s]/n1 - m1*m1
        if v1 < 1e-12: v1 = 1e-12
        v2 = (pz2[t]-pz2[s])/n2 - m2*m2
        if v2 < 1e-12: v2 = 1e-12
        dmean = (m2-m1)/math.sqrt(v1/n1 + v2/n2)
        if dmean < 0.0: dmean = -dmean
        dlvar = math.log(v2/v1)
        if dlvar < 0.0: dlvar = -dlvar
        mu3_1 = pz3[s]/n1 - 3.0*m1*v1 - m1*m1*m1
        mu3_2 = (pz3[t]-pz3[s])/n2 - 3.0*m2*v2 - m2*m2*m2
        dskew = mu3_2/(v2**1.5) - mu3_1/(v1**1.5)
        if dskew < 0.0: dskew = -dskew
        dtail = (pt[t]-pt[s])/n2 - pt[s]/n1
        if dtail < 0.0: dtail = -dtail
        if dmean > sm: sm = dmean
        if dlvar > sl: sl = dlvar
        if dskew > sk: sk = dskew
        if dtail > st: st = dtail
    return sm, sl, sk, st

@njit(cache=False)
def _cusum_at(z, start, n):
    cp = 0.0; cn = 0.0; mx = 0.0
    for i in range(n):
        v = z[start + i]
        a = cp + v - 0.5
        cp = a if a > 0.0 else 0.0
        b = cn - v - 0.5
        cn = b if b > 0.0 else 0.0
        c = cp if cp > cn else cn
        if c > mx: mx = c
    return mx

@njit(cache=False)
def _null_table(zh, widths):
    nw = widths.shape[0]
    out = np.zeros((nw, 10))
    n = zh.shape[0]
    for wi in range(nw):
        w = widths[wi]
        if n < 2 * w:
            out[wi, 1] = -1.0
            continue
        step = w // 2
        cnt = 0
        s1 = np.zeros(5); s2 = np.zeros(5)
        start = 0
        while start + w <= n:
            sm, sl, sk, st = _scan_stats_at(zh, start, w)
            cu = _cusum_at(zh, start, w)
            vals = np.array([sm, sl, sk, st, cu])
            for j in range(5):
                s1[j] += vals[j]; s2[j] += vals[j]*vals[j]
            cnt += 1
            start += step
        if cnt < 3:
            out[wi, 1] = -1.0
            continue
        for j in range(5):
            m = s1[j]/cnt
            v = s2[j]/cnt - m*m
            if v < 1e-12: v = 1e-12
            out[wi, 2*j] = m
            out[wi, 2*j+1] = math.sqrt(v)
    return out

@njit(cache=False)
def _null_curves(table, widths, n_o):
    out_m = np.zeros((n_o, 5)); out_s = np.ones((n_o, 5))
    nw = widths.shape[0]
    for ti in range(n_o):
        t = float(ti + 1)
        for j in range(5):
            lo_i = -1; hi_i = -1
            for i in range(nw):
                if table[i, 1] < 0.0: continue
                if widths[i] <= t: lo_i = i
                if widths[i] >= t and hi_i < 0: hi_i = i
            if lo_i < 0 and hi_i < 0:
                out_m[ti, j] = 0.0; out_s[ti, j] = -1.0
            elif lo_i < 0:
                out_m[ti, j] = table[hi_i, 2*j]; out_s[ti, j] = table[hi_i, 2*j+1]
            elif hi_i < 0 or lo_i == hi_i:
                out_m[ti, j] = table[lo_i, 2*j]; out_s[ti, j] = table[lo_i, 2*j+1]
            else:
                w0 = widths[lo_i]; w1 = widths[hi_i]
                f = (math.log(t) - math.log(w0)) / (math.log(w1) - math.log(w0))
                out_m[ti, j] = table[lo_i, 2*j]*(1-f) + table[hi_i, 2*j]*f
                out_s[ti, j] = table[lo_i, 2*j+1]*(1-f) + table[hi_i, 2*j+1]*f
    return out_m, out_s

@njit(cache=False)
def _interp_one(table, widths, t):
    m = np.zeros(5); s = np.zeros(5)
    nw = widths.shape[0]
    for j in range(5):
        lo_i = -1; hi_i = -1
        for i in range(nw):
            if table[i,1] < 0.0: continue
            if widths[i] <= t: lo_i = i
            if widths[i] >= t and hi_i < 0: hi_i = i
        if lo_i < 0 and hi_i < 0:
            m[j] = 0.0; s[j] = -1.0
        elif lo_i < 0:
            m[j] = table[hi_i,2*j]; s[j] = table[hi_i,2*j+1]
        elif hi_i < 0 or lo_i == hi_i:
            m[j] = table[lo_i,2*j]; s[j] = table[lo_i,2*j+1]
        else:
            w0 = widths[lo_i]; w1 = widths[hi_i]
            f = (math.log(t)-math.log(w0))/(math.log(w1)-math.log(w0))
            m[j] = table[lo_i,2*j]*(1-f) + table[hi_i,2*j]*f
            s[j] = table[lo_i,2*j+1]*(1-f) + table[hi_i,2*j+1]*f
    return m, s

def calib_block(stream_hist, F):
    """Vectorised: used by train()."""
    W = np.array([50, 100, 200, 400, 800], dtype=np.int64)
    z = np.asarray(stream_hist, dtype=np.float64)
    mu = float(z.mean()); sd = max(float(z.std(ddof=1)), 1e-8)
    zh = (z - mu) / sd
    table = _null_table(zh, W)
    n_o = F.shape[0]
    m, s = _null_curves(table, W.astype(np.float64), n_o)
    raw = np.empty((n_o, 5))
    raw[:, 0] = F[:, 22]; raw[:, 1] = F[:, 23]
    raw[:, 2] = F[:, 24]; raw[:, 3] = F[:, 25]; raw[:, 4] = F[:, 1]
    out = np.zeros((n_o, 5))
    ok = s > 0
    out[ok] = (raw[ok] - m[ok]) / np.maximum(s[ok], 1e-6)
    np.clip(out, -20.0, 20.0, out=out)
    return out

def make_null(stream_hist):
    """Per-series null table: used by infer()."""
    W = np.array([50, 100, 200, 400, 800], dtype=np.int64)
    z = np.asarray(stream_hist, dtype=np.float64)
    mu = float(z.mean()); sd = max(float(z.std(ddof=1)), 1e-8)
    zh = (z - mu) / sd
    return _null_table(zh, W)

def calib_point(table, t, sm, sl, sk, st, cu):
    """Per-point: used by infer(). Matches calib_block to 2e-14."""
    W = np.array([50.0, 100.0, 200.0, 400.0, 800.0], dtype=np.float64)
    m, s = _interp_one(table, W, float(t))
    raw = (sm, sl, sk, st, cu)
    out = [0.0] * 5
    for j in range(5):
        if s[j] > 0:
            v = (raw[j] - m[j]) / max(s[j], 1e-6)
            if v > 20.0: v = 20.0
            elif v < -20.0: v = -20.0
            out[j] = v
    return out


# ═══════════════════════════════════════════════════════════
# CELL 3b of 6 — ACCUMULATOR CALIBRATION
# tail_frac(3), ll_norm(26), r_cusum(31), g_cusum_n(33)
# Widths defined inside functions (converter strips top-level).
# ═══════════════════════════════════════════════════════════
import numpy as np, math
from numba import njit

@njit(cache=False)
def _acc_stats_at(zw, uw, start, n):
    n_tail = 0; ll_cum = 0.0; r_cusum = 0.0
    ew_var = 1.0; g_cusum = 0.0
    for i in range(n):
        z = zw[start + i]
        az = z if z >= 0.0 else -z
        if az > 2.0: n_tail += 1
        u = uw[start + i]
        ll_cum += 0.5 * (u * u - 1.0)
        rc = r_cusum + u - 0.5
        r_cusum = rc if rc > 0.0 else 0.0
        evd = ew_var if ew_var > 1e-6 else 1e-6
        g = u / math.sqrt(evd)
        ew_var = 0.94 * ew_var + 0.06 * u * u
        ag = g if g >= 0.0 else -g
        gc = g_cusum + ag - 0.8
        g_cusum = gc if gc > 0.0 else 0.0
    return n_tail / n, ll_cum / math.sqrt(n), r_cusum, g_cusum / math.sqrt(n)

@njit(cache=False)
def _null_table_acc(zh, uh, widths):
    nw = widths.shape[0]
    out = np.zeros((nw, 8))
    n = zh.shape[0]
    for wi in range(nw):
        w = widths[wi]
        if n < 2 * w:
            out[wi, 1] = -1.0
            continue
        step = w // 2
        cnt = 0
        s1 = np.zeros(4); s2 = np.zeros(4)
        start = 0
        while start + w <= n:
            a, b, c, d = _acc_stats_at(zh, uh, start, w)
            vals = np.array([a, b, c, d])
            for j in range(4):
                s1[j] += vals[j]; s2[j] += vals[j] * vals[j]
            cnt += 1
            start += step
        if cnt < 3:
            out[wi, 1] = -1.0
            continue
        for j in range(4):
            m = s1[j] / cnt
            v = s2[j] / cnt - m * m
            if v < 1e-12: v = 1e-12
            out[wi, 2*j] = m
            out[wi, 2*j+1] = math.sqrt(v)
    return out

@njit(cache=False)
def _null_curves_acc(table, widths, n_o):
    out_m = np.zeros((n_o, 4)); out_s = np.ones((n_o, 4))
    nw = widths.shape[0]
    for ti in range(n_o):
        t = float(ti + 1)
        for j in range(4):
            lo_i = -1; hi_i = -1
            for i in range(nw):
                if table[i, 1] < 0.0: continue
                if widths[i] <= t: lo_i = i
                if widths[i] >= t and hi_i < 0: hi_i = i
            if lo_i < 0 and hi_i < 0:
                out_m[ti, j] = 0.0; out_s[ti, j] = -1.0
            elif lo_i < 0:
                out_m[ti, j] = table[hi_i, 2*j]; out_s[ti, j] = table[hi_i, 2*j+1]
            elif hi_i < 0 or lo_i == hi_i:
                out_m[ti, j] = table[lo_i, 2*j]; out_s[ti, j] = table[lo_i, 2*j+1]
            else:
                w0 = widths[lo_i]; w1 = widths[hi_i]
                f = (math.log(t) - math.log(w0)) / (math.log(w1) - math.log(w0))
                out_m[ti, j] = table[lo_i, 2*j]*(1-f) + table[hi_i, 2*j]*f
                out_s[ti, j] = table[lo_i, 2*j+1]*(1-f) + table[hi_i, 2*j+1]*f
    return out_m, out_s

@njit(cache=False)
def _interp_one_acc(table, widths, t):
    m = np.zeros(4); s = np.zeros(4)
    nw = widths.shape[0]
    for j in range(4):
        lo_i = -1; hi_i = -1
        for i in range(nw):
            if table[i,1] < 0.0: continue
            if widths[i] <= t: lo_i = i
            if widths[i] >= t and hi_i < 0: hi_i = i
        if lo_i < 0 and hi_i < 0:
            m[j] = 0.0; s[j] = -1.0
        elif lo_i < 0:
            m[j] = table[hi_i,2*j]; s[j] = table[hi_i,2*j+1]
        elif hi_i < 0 or lo_i == hi_i:
            m[j] = table[lo_i,2*j]; s[j] = table[lo_i,2*j+1]
        else:
            w0 = widths[lo_i]; w1 = widths[hi_i]
            f = (math.log(t)-math.log(w0))/(math.log(w1)-math.log(w0))
            m[j] = table[lo_i,2*j]*(1-f) + table[hi_i,2*j]*f
            s[j] = table[lo_i,2*j+1]*(1-f) + table[hi_i,2*j+1]*f
    return m, s

def _hist_streams(stream_hist, ar_order):
    x = np.asarray(stream_hist, dtype=np.float64)
    mu = float(x.mean()); sd = max(float(x.std(ddof=1)), 1e-8)
    zh = (x - mu) / sd
    p = ar_order
    r = _fit_ar_p(zh, p)
    if r is None:
        p, phi, rv = _select_ar_order(zh, 4)
    else:
        phi, rv, _ = r
    srv = math.sqrt(rv)
    n = len(zh)
    uh = np.empty(n)
    for i in range(n):
        pred = 0.0
        for j in range(min(i, p)):
            pred += phi[j] * zh[i - 1 - j]
        uh[i] = (zh[i] - pred) / srv
    return zh, uh

def make_null_acc(stream_hist, ar_order):
    W = np.array([50, 100, 200, 400, 800], dtype=np.int64)
    zh, uh = _hist_streams(stream_hist, ar_order)
    return _null_table_acc(zh, uh, W)

def calib_block_acc(stream_hist, F, ar_order):
    """Vectorised: used by train()."""
    W = np.array([50, 100, 200, 400, 800], dtype=np.int64)
    table = make_null_acc(stream_hist, ar_order)
    n_o = F.shape[0]
    m, s = _null_curves_acc(table, W.astype(np.float64), n_o)
    raw = np.empty((n_o, 4))
    raw[:, 0] = F[:, 3]; raw[:, 1] = F[:, 26]
    raw[:, 2] = F[:, 31]; raw[:, 3] = F[:, 33]
    out = np.zeros((n_o, 4))
    ok = s > 0
    out[ok] = (raw[ok] - m[ok]) / np.maximum(s[ok], 1e-6)
    np.clip(out, -20.0, 20.0, out=out)
    return out

def calib_point_acc(table, t, tf, ln, rc, gc):
    """Per-point: used by infer(). Matches calib_block_acc exactly."""
    W = np.array([50.0, 100.0, 200.0, 400.0, 800.0], dtype=np.float64)
    m, s = _interp_one_acc(table, W, float(t))
    raw = (tf, ln, rc, gc)
    out = [0.0] * 4
    for j in range(4):
        if s[j] > 0:
            v = (raw[j] - m[j]) / max(s[j], 1e-6)
            if v > 20.0: v = 20.0
            elif v < -20.0: v = -20.0
            out[j] = v
    return out

loaded crunch tools for module: <module '__main__'>

cli version: 12.0.2
available ram: 12.67 gb
available cpu: 2 core
----
data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_train.parquet (8356193 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_test.reduced.parquet (106299 bytes)
data/y_test.reduced.parquet: already exists, file le

In [3]:
import numpy as np, math
from numba import njit

def _sr_consts():
    M = np.array([4,6,8,12,16,24,32,48,64,96,128,192,256,384,512,768], dtype=np.int64)
    LOGW = np.log(np.diff(np.concatenate((np.array([0], dtype=np.int64), M))).astype(np.float64))
    A = np.array([100.0, 30.0, 12.0, 6.0, 3.0])
    B = A - 1.0
    SD = np.array([0.064, 0.154, 0.384])
    R = np.array([0.067, -0.067, 0.134, -0.134, 0.224, -0.224, 0.392, -0.392])
    return M, LOGW, A, B, SD, R


@njit(cache=False)
def _lse_add(best, acc, v, first):
    if first:
        return v, 1.0
    if v > best:
        return v, acc * math.exp(best - v) + 1.0
    return best, acc + math.exp(v - best)


@njit(cache=False)
def _sr_point(T, c2, c1, cx, cp, M, LOGW, A, B, SD, R):
    nm = M.shape[0]
    ln = math.log(float(T))

    b_s = -1e300; a_s = 0.0; f_s = True
    b_m = -1e300; a_m = 0.0; f_m = True
    b_d = -1e300; a_d = 0.0; f_d = True

    for mi in range(nm):
        m = M[mi]
        if m > T:
            break
        w = LOGW[mi]
        s2 = c2[T] - c2[T - m]
        s1 = c1[T] - c1[T - m]
        sx = cx[T] - cx[T - m]
        sp = cp[T] - cp[T - m]

        for ai in range(A.shape[0]):
            a = A[ai]; b = B[ai]
            v = (a * math.log(b) - math.lgamma(a) + math.lgamma(a + 0.5 * m)
                 - (a + 0.5 * m) * math.log(b + 0.5 * s2) + 0.5 * s2
                 + w - math.log(float(A.shape[0])))
            b_s, a_s = _lse_add(b_s, a_s, v, f_s); f_s = False

        for si in range(SD.shape[0]):
            s = SD[si] * SD[si]
            v = (-0.5 * math.log(1.0 + m * s) + s1 * s1 * s / (2.0 * (1.0 + m * s))
                 + w - math.log(float(SD.shape[0])))
            b_m, a_m = _lse_add(b_m, a_m, v, f_m); f_m = False

        for ri in range(R.shape[0]):
            r = R[ri]
            q = 1.0 - r * r
            v = (-0.5 * (s2 - 2.0 * r * sx + r * r * sp) / q - 0.5 * m * math.log(q)
                 + 0.5 * s2 + w - math.log(float(R.shape[0])))
            b_d, a_d = _lse_add(b_d, a_d, v, f_d); f_d = False

    o_s = 0.0 if f_s else b_s + math.log(a_s) - ln
    o_m = 0.0 if f_m else b_m + math.log(a_m) - ln
    o_d = 0.0 if f_d else b_d + math.log(a_d) - ln
    return o_s, o_m, o_d


class MixtureSR:
    __slots__ = ("c2","c1","cx","cp","T","prev","cap","M","LOGW","A","B","SD","R")

    def __init__(self, cap=1200):
        self.cap = cap
        self.c2 = np.zeros(cap + 1); self.c1 = np.zeros(cap + 1)
        self.cx = np.zeros(cap + 1); self.cp = np.zeros(cap + 1)
        self.T = 0; self.prev = 0.0
        self.M, self.LOGW, self.A, self.B, self.SD, self.R = _sr_consts()

    def _grow(self):
        n = self.cap * 2
        for a in ("c2", "c1", "cx", "cp"):
            old = getattr(self, a)
            new = np.zeros(n + 1); new[:len(old)] = old
            setattr(self, a, new)
        self.cap = n

    def step(self, u):
        if self.T + 1 > self.cap:
            self._grow()
        t = self.T + 1
        self.c2[t] = self.c2[t-1] + u * u
        self.c1[t] = self.c1[t-1] + u
        self.cx[t] = self.cx[t-1] + u * self.prev
        self.cp[t] = self.cp[t-1] + self.prev * self.prev
        self.T = t; self.prev = u
        return _sr_point(t, self.c2, self.c1, self.cx, self.cp,
                         self.M, self.LOGW, self.A, self.B, self.SD, self.R)


def mixsr_batch(u):
    s = MixtureSR()
    out = np.empty((len(u), 3))
    for i in range(len(u)):
        out[i, 0], out[i, 1], out[i, 2] = s.step(u[i])
    return out

**train**

In [4]:
import numpy as np
from catboost import CatBoostClassifier
from scipy.stats import norm as _norm
from numba import njit
import gc, math

# @crunch/keep:on
INFER_PARALLELISM = 16

_LOOKUP_SIZE = 100_001
_LOOKUP_G = _norm.ppf(
    np.linspace(1.0 / _LOOKUP_SIZE, 1.0 - 1.0 / _LOOKUP_SIZE, _LOOKUP_SIZE)
).astype(np.float64)

def _rank_to_gauss(rank, n):
    p = rank / (n + 1)
    if p < 1.0 / (n + 1): p = 1.0 / (n + 1)
    elif p > n / (n + 1): p = n / (n + 1)
    return float(_LOOKUP_G[int(p * (_LOOKUP_SIZE - 1))])

def _rank_history(h_sorted, values):
    n = len(h_sorted)
    ranks = np.searchsorted(h_sorted, values, side="right")
    p = ranks / (n + 1.0)
    np.clip(p, 1.0 / (n + 1.0), n / (n + 1.0), out=p)
    idx = (p * (_LOOKUP_SIZE - 1)).astype(np.int64)
    return _LOOKUP_G[idx]

def _ar_setup(h):
    mu = float(h.mean()); sd = max(float(h.std(ddof=1)), 1e-9)
    zh = (h - mu) / sd
    p, phi, rv = _select_ar_order(zh, 4)
    srv = math.sqrt(rv)
    n = len(zh)
    resid = np.empty(n, dtype=np.float64)
    for i in range(n):
        pred = 0.0
        for j in range(min(i, p)):
            pred += phi[j] * zh[i - 1 - j]
        resid[i] = (zh[i] - pred) / srv
    return mu, sd, p, np.asarray(phi, dtype=np.float64), srv, resid

def _vol_init(resid_h):
    ew_f = 1.0; ew_s = 1.0
    for u in resid_h:
        ew_f = 0.94 * ew_f + 0.06 * u * u
        ew_s = 0.995 * ew_s + 0.005 * u * u
    return ew_f, ew_s

@njit(cache=False)
def _resid_stream(z_all, phi, srv, lags0, p):
    n = z_all.shape[0]
    u = np.empty(n)
    lags = lags0.copy()
    for t in range(n):
        z = z_all[t]
        pred = 0.0
        for j in range(p):
            pred += phi[j] * lags[j]
        u[t] = (z - pred) / srv
        if p > 0:
            for j in range(p - 1, 0, -1):
                lags[j] = lags[j - 1]
            lags[0] = z
    return u

@njit(cache=False)
def _ewma_pair(u, ew_f0, ew_s0):
    n = u.shape[0]
    a = np.empty(n); b = np.empty(n)
    ef = ew_f0; es = ew_s0
    for t in range(n):
        ef = 0.94 * ef + 0.06 * u[t] * u[t]
        es = 0.995 * es + 0.005 * u[t] * u[t]
        a[t] = math.log(ef if ef > 1e-10 else 1e-10)
        b[t] = math.log((ef if ef > 1e-10 else 1e-10) / (es if es > 1e-10 else 1e-10))
    return a, b


def train(datasets, model_directory_path):
    import os, time

    MODEL_VERSION = "mixsr195-v1"
    mpath = os.path.join(model_directory_path, "model.joblib")
    if os.path.exists(mpath):
        try:
            existing = joblib.load(mpath)
            if existing.get("version") == MODEL_VERSION:
                print(f"pre-trained model found ({MODEL_VERSION}) — skipping training")
                return
        except Exception as e:
            print(f"existing model unreadable ({e}) — retraining")

    datasets = list(datasets)
    t0 = time.time()

    row_blocks, label_blocks = [], []
    for k, (dataset_id, x_hist, x_online, tau) in enumerate(datasets):
        h = np.asarray(x_hist, dtype=np.float64)
        o = np.asarray(x_online, dtype=np.float64)
        n_o = len(o)

        mu, sd, p, phi, srv, resid_h = _ar_setup(h)
        z_all = (o - mu) / sd
        zh = (h - mu) / sd
        lags0 = np.zeros(max(p, 1))
        for j in range(p):
            lags0[j] = zh[-1 - j] if len(zh) > j else 0.0
        u_stream = _resid_stream(z_all, phi, srv, lags0, p)

        SR = mixsr_batch(u_stream)

        h_sorted = np.sort(h)
        h_ranked = _rank_history(h_sorted, h)
        o_ranked = _rank_history(h_sorted, o)
        abs_h = np.abs(resid_h); abs_mu = float(abs_h.mean())
        abs_hist = abs_h - abs_mu

        F_raw  = StreamingFeaturesJIT(h, ar_order=p).run(o)
        F_rank = StreamingFeaturesJIT(h_ranked, ar_order=p).run(o_ranked)
        F_res  = StreamingFeaturesJIT(resid_h, ar_order=p).run(u_stream)
        F_abs  = StreamingFeaturesJIT(abs_hist, ar_order=p).run(np.abs(u_stream) - abs_mu)

        C_raw  = calib_block(h, F_raw)
        C_rank = calib_block(h_ranked, F_rank)
        C_res  = calib_block(resid_h, F_res)
        C_abs  = calib_block(abs_hist, F_abs)

        A_raw  = calib_block_acc(h, F_raw, p)
        A_rank = calib_block_acc(h_ranked, F_rank, p)
        A_res  = calib_block_acc(resid_h, F_res, p)
        A_abs  = calib_block_acc(abs_hist, F_abs, p)

        ew_f0, ew_s0 = _vol_init(resid_h)
        v_fast, v_ratio = _ewma_pair(u_stream, ew_f0, ew_s0)

        tar50 = _TrailingAR(p, 50, phi, srv * srv)
        tar150 = _TrailingAR(p, 150, phi, srv * srv)
        tar300 = _TrailingAR(p, 300, phi, srv * srv)
        T = np.empty((n_o, 6))
        for t in range(n_o):
            z = z_all[t]
            T[t, 0], T[t, 1] = tar50.update(z)
            T[t, 2], T[t, 3] = tar150.update(z)
            T[t, 4], T[t, 5] = tar300.update(z)

        blk = np.empty((n_o, 195), dtype=np.float32)
        blk[:, :37] = F_raw
        blk[:, 37:74] = F_rank
        blk[:, 74:111] = F_res
        blk[:, 111:148] = F_abs
        blk[:, 148] = v_fast
        blk[:, 149] = v_ratio
        blk[:, 150:156] = T
        blk[:, 156:161] = C_raw
        blk[:, 161:166] = C_rank
        blk[:, 166:171] = C_res
        blk[:, 171:176] = C_abs
        blk[:, 176:180] = A_raw
        blk[:, 180:184] = A_rank
        blk[:, 184:188] = A_res
        blk[:, 188:192] = A_abs
        blk[:, 192:195] = SR

        lab = np.zeros(n_o, dtype=np.int8)
        if tau is not None:
            lab[int(tau):] = 1

        row_blocks.append(blk)
        label_blocks.append(lab)

        if (k + 1) % 2000 == 0:
            print(f"  features: {k+1}/{len(datasets)} series ({(time.time()-t0)/60:.0f} min)")

    X = np.concatenate(row_blocks, axis=0); del row_blocks
    y = np.concatenate(label_blocks, axis=0); del label_blocks
    gc.collect()
    print(f"features built: {X.shape} ({(time.time()-t0)/60:.0f} min)")

    SEEDS = (42, 123, 777)
    cats = []
    for s in SEEDS:
        c = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                               random_seed=s, verbose=0, allow_writing_files=False,
                               thread_count=8)
        c.fit(X, y)
        cats.append(c)
        print(f"seed {s} fitted ({(time.time()-t0)/60:.0f} min)")

    del X, y; gc.collect()

    joblib.dump({"cats": cats, "version": MODEL_VERSION},
                os.path.join(model_directory_path, "model.joblib"))
    print(f"model saved ({(time.time()-t0)/60:.0f} min total)")

**infer**

In [5]:
import numpy as np

def infer(datasets, model_directory_path):
    import os, math

    bundle = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    cats = bundle["cats"]
    n_cat = len(cats)

    yield  # readiness

    for x_historical, x_online in datasets:
        h_arr = np.asarray(x_historical, dtype=np.float64)

        mu, sd, p, phi, srv, resid_h = _ar_setup(h_arr)

        h_sorted = np.sort(h_arr)
        n_h = len(h_sorted)
        h_ranked = _rank_history(h_sorted, h_arr)
        abs_h = np.abs(resid_h)
        abs_mu = float(abs_h.mean())
        abs_hist = abs_h - abs_mu

        sf_raw = StreamingFeatures(h_arr, ar_order=p)
        sf_rank = StreamingFeatures(h_ranked, ar_order=p)
        sf_res = StreamingFeatures(resid_h, ar_order=p)
        sf_abs = StreamingFeatures(abs_hist, ar_order=p)

        nt_raw = make_null(h_arr)
        nt_rank = make_null(h_ranked)
        nt_res = make_null(resid_h)
        nt_abs = make_null(abs_hist)

        na_raw = make_null_acc(h_arr, p)
        na_rank = make_null_acc(h_ranked, p)
        na_res = make_null_acc(resid_h, p)
        na_abs = make_null_acc(abs_hist, p)

        tar50 = _TrailingAR(p, 50, phi, srv * srv)
        tar150 = _TrailingAR(p, 150, phi, srv * srv)
        tar300 = _TrailingAR(p, 300, phi, srv * srv)

        ew_f, ew_s = _vol_init(resid_h)
        sr = MixtureSR()

        zh = (h_arr - mu) / sd
        lags = [0.0] * max(p, 1)
        for j in range(p):
            lags[j] = zh[-1 - j] if len(zh) > j else 0.0

        peak = 0.0
        for point in x_online:
            pt = float(point)
            f_raw = sf_raw.update(pt)
            rank = int(np.searchsorted(h_sorted, pt, side="right"))
            f_rank = sf_rank.update(_rank_to_gauss(rank, n_h))
            z = (pt - mu) / sd
            pred = 0.0
            for j in range(p):
                pred += phi[j] * lags[j]
            u = (z - pred) / srv
            if p > 0:
                for j in range(p - 1, 0, -1):
                    lags[j] = lags[j-1]
                lags[0] = z
            f_res = sf_res.update(u)
            f_abs = sf_abs.update(abs(u) - abs_mu)

            sr_s, sr_m, sr_d = sr.step(u)

            d50, lv50 = tar50.update(z)
            d150, lv150 = tar150.update(z)
            d300, lv300 = tar300.update(z)

            t_now = f_raw[0]
            c_raw = calib_point(nt_raw, t_now, f_raw[22], f_raw[23],
                                f_raw[24], f_raw[25], f_raw[1])
            c_rank = calib_point(nt_rank, t_now, f_rank[22], f_rank[23],
                                 f_rank[24], f_rank[25], f_rank[1])
            c_res = calib_point(nt_res, t_now, f_res[22], f_res[23],
                                f_res[24], f_res[25], f_res[1])
            c_abs = calib_point(nt_abs, t_now, f_abs[22], f_abs[23],
                                f_abs[24], f_abs[25], f_abs[1])

            a_raw = calib_point_acc(na_raw, t_now, f_raw[3], f_raw[26],
                                    f_raw[31], f_raw[33])
            a_rank = calib_point_acc(na_rank, t_now, f_rank[3], f_rank[26],
                                     f_rank[31], f_rank[33])
            a_res = calib_point_acc(na_res, t_now, f_res[3], f_res[26],
                                    f_res[31], f_res[33])
            a_abs = calib_point_acc(na_abs, t_now, f_abs[3], f_abs[26],
                                    f_abs[31], f_abs[33])

            ew_f = 0.94 * ew_f + 0.06 * u * u
            ew_s = 0.995 * ew_s + 0.005 * u * u
            v_fast = math.log(max(ew_f, 1e-10))
            v_ratio = math.log(max(ew_f, 1e-10) / max(ew_s, 1e-10))

            row = np.array([f_raw + f_rank + f_res + f_abs +
                            (v_fast, v_ratio, d50, lv50, d150, lv150, d300, lv300) +
                            tuple(c_raw) + tuple(c_rank) + tuple(c_res) + tuple(c_abs) +
                            tuple(a_raw) + tuple(a_rank) + tuple(a_res) + tuple(a_abs) +
                            (sr_s, sr_m, sr_d)],
                           dtype=np.float64)

            s = 0.0
            for c in cats:
                s += float(c.predict_proba(row, thread_count=1)[0, 1])
            s /= n_cat

            if s > peak: peak = s
            yield 0.2 * peak + 0.8 * s

In [6]:
_, h, o, tau = train_data[0]
h = np.asarray(h, np.float64); o = np.asarray(o, np.float64)
mu, sd, p, phi, srv, rh = _ar_setup(h)
zz = (o-mu)/sd; zh = (h-mu)/sd
l0 = np.zeros(max(p,1))
for j in range(p): l0[j] = zh[-1-j] if len(zh) > j else 0.0
uu = _resid_stream(zz, phi, srv, l0, p)

A = mixsr_batch(uu)
st = MixtureSR()
B = np.array([st.step(v) for v in uu])
print("diff:", np.abs(A-B).max(), " shape:", A.shape)

diff: 0.0  shape: (416, 3)


In [7]:
!rm -f resources/model.joblib screen176.npz screen156.npz
!cat requirements.txt

# extracted from a notebook

## third-party
catboost
crunch-cli  # alias of crunch
joblib
numba
numpy
pandas
scikit-learn  # alias of sklearn
scipy

## standard
#collections
#gc
#math
#os
#typing


In [8]:
# @title  {"display-mode":"form", "form-width":"400px"}

# @markdown Describe your changes, then run the cell.
Message = "" # @param {"type":"string","placeholder":"Short description (optional)"}

# ---
# THIS METHOD IS ONLY POSSIBLE ON COLAB.
# RUNNING THIS CELL WILL PROMPT YOU TO USE THE OLD WAY OF SUBMITTING A NOTEBOOK.

crunch_tools.submit(
    message=Message,
)

warning ogpg-Og0u8t0: line 80: column 0: nested import: found 1 nested import in FunctionDef statement
warning BEq36II7u8xQ: line 3: column 0: nested import: found 1 nested import in FunctionDef statement


found code file: main.py (51.19 KB)


uploading `main.py`:   0%|          | 0.00/50.0k [00:00<?, ?B/s]

found code file: requirements.txt (196 bytes)


uploading `requirements.txt`:   0%|          | 0.00/196 [00:00<?, ?B/s]

found code file: notebook.ipynb (66.32 KB)


uploading `notebook.ipynb`:   0%|          | 0.00/64.8k [00:00<?, ?B/s]

total code size: 117.71 KB
total model size: 0 bytes
export structural-break-real-time:project/13312/breakyy



---

Next step is to run your submission in the cloud:

### >> https://hub.crunchdao.com/competitions/structural-break-real-time/models/pixelated-anja/breakyy/runs/create?submissionNumber=169

<img alt="Run in the Cloud" src="https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/create-run.gif" height="600px" />
